In [21]:
%load_ext autoreload
%autoreload 2
import sisl
import numpy as np
import matplotlib.pyplot as plt
from mytools.plots import thesis_fig
from scipy.spatial import cKDTree
from mytools.construct import all_armchair
from ase.visualize.plot import plot_atoms
from tqdm.auto import tqdm


from mytools.scalingv2 import get_centers, get_corners, get_edges, get_fractional
from mytools.scalingv2 import rsse_to_edge, rsse_mapping
from mytools.construct import make_edge


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [9]:
BOND = 1.42
phi = np.pi/3
B = BOND*np.cos(phi/2)

N0 = 4
N_small = 7
N_big = 12

R = (0.1, BOND+1e-2)
T = (0.0, -2.7)

ETA = 1e-3
NUMK = 400

In [24]:
graphene6 = all_armchair(BOND)

Ham0 = sisl.Hamiltonian(graphene6)
Ham0.construct([R, T])

# Make the edges and RSSE

In [25]:
rsse_small = sisl.RealSpaceSE(Ham0, 0, 1, (N_small, N_small, 1))
rsse_small.setup(eta=ETA, 
            bz=sisl.MonkhorstPack(Ham0, [1, NUMK, 1]))

rsse_big = sisl.RealSpaceSE(Ham0, 0, 1, (N_big, N_big, 1))
rsse_big.setup(eta=ETA, 
            bz=sisl.MonkhorstPack(Ham0, [1, NUMK, 1]))


In [34]:
def resub_ham(rsse):
    Ham_rs, elec_idx = rsse.real_space_coupling(ret_indices=True)
    Ham_NN = rsse.real_space_parent()
    all_idx = np.arange(Ham_NN.na)
    device_idx = np.delete(all_idx, elec_idx)
    sub_idx = np.concat([elec_idx, device_idx])
    Ham_NN_re = Ham_NN.sub(sub_idx)
    
    out = {
        "H_elec": Ham_rs,
        "elec_idx": elec_idx,
        "sub_idx": sub_idx,
        
    }
    return Ham_NN_re, out

In [ ]:
Ham_reord_small, d_small = resub_ham(rsse_small)
Ham_reord_big, d_big = resub_ham(rsse_big)

elec_idx_small = d_small['elec_idx']
elec_idx_big = d_big['elec_idx']

H_elec_small = d_small['H_elec']
H_elec_big = d_big["H_elec"]

sub_idx_small = d_small['sub_idx']
sub_idx_big = d_big['sub_idx']

In [27]:
geom_edge_small = make_edge(graphene6, N_small, N_small)
geom_edge_big = make_edge(graphene6, N_big, N_big)

In [28]:
NC = 1
na = Ham0.na
corners_small = get_corners(N_small, N_small, na=na, NC=NC)
corners_big = get_corners(N_big, N_big, na=na, NC=NC)
edges_small = get_edges(N_small, N_small, na=na, NC=NC)
edges_big = get_edges(N_big, N_big, na=na, NC=NC)